In [29]:
import spacy # importando a biblioteca spacy
!python -m spacy download en_core_web_md # baixando o modelo médio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 26.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [30]:
nlp = spacy.load("en_core_web_md") # carregando o modelo
text = "Paragon defeated the Lich King using Bloodlust while Lazeil tanked the Shambling Horrors on Icecrown Citadel."
doc = nlp(text) # processando o texto

In [31]:
for ent in doc.ents: # verificando o que o modelo reconhece
    print(ent.text, ent.label_)

Paragon ORG
Lazeil PERSON
Icecrown Citadel PERSON


In [32]:
# o modelo não conhece o universo WoW ent adicionei um entitu_rule depois do ner para ensinar ele

ruler = nlp.add_pipe("entity_ruler") # adicionando o ruler ao final
nlp.analyze_pipes() # confirmando a posição do ruler

{'summary': {'tok2vec': {'assigns': ['doc.tensor'],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'tagger': {'assigns': ['token.tag'],
   'requires': [],
   'scores': ['tag_acc',
    'pos_acc',
    'tag_micro_p',
    'tag_micro_r',
    'tag_micro_f'],
   'retokenizes': False},
  'parser': {'assigns': ['token.dep',
    'token.head',
    'token.is_sent_start',
    'doc.sents'],
   'requires': [],
   'scores': ['dep_uas',
    'dep_las',
    'dep_las_per_type',
    'sents_p',
    'sents_r',
    'sents_f'],
   'retokenizes': False},
  'attribute_ruler': {'assigns': [],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'lemmatizer': {'assigns': ['token.lemma'],
   'requires': [],
   'scores': ['lemma_acc'],
   'retokenizes': False},
  'ner': {'assigns': ['doc.ents', 'token.ent_iob', 'token.ent_type'],
   'requires': [],
   'scores': ['ents_f', 'ents_p', 'ents_r', 'ents_per_type'],
   'retokenizes': False},
  'entity_ruler': {'assigns': ['doc.ents', 'token.ent_

In [38]:
patterns = [
    {"label": "Guilda", "pattern": "Paragon"}, # guildas famosas do WoW
    {"label": "Boss", "pattern": "Lich King"}, # chefes finais de Icecrown
    {"label": "Habilidade", "pattern": "Bloodlust"},# habilidade do Shaman
    {"label": "NPC", "pattern": "Shambling Horrors"}, # adds do encontro
    {"label": "Localizacao", "pattern": "Icecrown Citadel"} # raid onde ocorre
]

ruler.add_patterns(patterns) # injetando os padroes

In [39]:
doc2 = nlp(text) # reprocessando o texto e vê se foi aplicado

for ent in doc2.ents: # verificando as entidades
    print(ent.text, ent.label_)

Paragon ORG
Lich King Boss
Bloodlust Habilidade
Lazeil PERSON
Shambling Horrors NPC
Icecrown Citadel PERSON


In [40]:
nlp2 = spacy.load("en_core_web_md") # carregando um modelo limpo
ruler2 = nlp2.add_pipe("entity_ruler", before="ner") # ruler antes do ner

nlp2.analyze_pipes() # confirmando a nova ordem

{'summary': {'tok2vec': {'assigns': ['doc.tensor'],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'tagger': {'assigns': ['token.tag'],
   'requires': [],
   'scores': ['tag_acc',
    'pos_acc',
    'tag_micro_p',
    'tag_micro_r',
    'tag_micro_f'],
   'retokenizes': False},
  'parser': {'assigns': ['token.dep',
    'token.head',
    'token.is_sent_start',
    'doc.sents'],
   'requires': [],
   'scores': ['dep_uas',
    'dep_las',
    'dep_las_per_type',
    'sents_p',
    'sents_r',
    'sents_f'],
   'retokenizes': False},
  'attribute_ruler': {'assigns': [],
   'requires': [],
   'scores': [],
   'retokenizes': False},
  'lemmatizer': {'assigns': ['token.lemma'],
   'requires': [],
   'scores': ['lemma_acc'],
   'retokenizes': False},
  'entity_ruler': {'assigns': ['doc.ents', 'token.ent_type', 'token.ent_iob'],
   'requires': [],
   'scores': ['ents_f', 'ents_p', 'ents_r', 'ents_per_type'],
   'retokenizes': False},
  'ner': {'assigns': ['doc.ents', 'token.ent_

In [41]:
patterns2 = [
    {"label": "Guilda", "pattern": "Paragon"},
    {"label": "boss", "pattern": "Lich King"},
    {"label": "habilidades", "pattern": "Bloodlust"},
    {"label": "Tank", "pattern": "Lazeil"},
    {"label": "NPC", "pattern": "Shambling Horrors"},
    {"label": "localizacao", "pattern": "Icecrown Citadel"}
]

ruler2.add_patterns(patterns2)

In [42]:
doc3 = nlp2(text) # processando o ruler

for ent in doc3.ents: # comparando o resultado com o da célula 6
    print(ent.text, ent.label_)

Paragon Guilda
Lich King boss
Bloodlust habilidades
Lazeil Tank
Shambling Horrors NPC
Icecrown Citadel localizacao


In [43]:
# testando com o texto completo
with open("historico_wow.txt", "r") as f:
    full_text = f.read()

doc4 = nlp2(full_text)

for ent in doc4.ents: # exibindo todas as entidades
    print(ent.text, ent.label_)

the World First Kill of Heroic ORG
25 CARDINAL
Lich King boss
Paragon Guilda
Raid Composition PERSON
3 x QUANTITY
1 CARDINAL
4 CARDINAL
2 CARDINAL
1 CARDINAL
2 CARDINAL
5 CARDINAL
Paladin - 1 Protection ORG
2 Holy FAC
2 CARDINAL
3 CARDINAL
1 CARDINAL
1 CARDINAL
3 CARDINAL
2 CARDINAL
1 CARDINAL
3 CARDINAL
2 CARDINAL
1 CARDINAL
25 CARDINAL
Lich King boss
Paragon Guilda
Lich King boss
That week DATE
5% PERCENT
buff PERSON
the previous Wednesday DATE
Frost DK GPE
Boomkin PERSON
5% PERCENT
Lich King boss
103.9 CARDINAL
10% PERCENT
Val'kyr Shadowguard ORG
3 CARDINAL
4.1 CARDINAL
3 CARDINAL
2 CARDINAL
the Frostmourne Room FAC
1 CARDINAL
2 CARDINAL
Shambling Horrors NPC
Ghouls ORG
Lazeil Tank
Lich King boss
Ghouls ORG
GHOULS ORG
Lazeil Tank
Lazeil Tank
Paladin PERSON
Lazeil Tank
two CARDINAL
Enrage ORG
two CARDINAL
20% PERCENT
1 CARDINAL
two CARDINAL
2 CARDINAL
Two CARDINAL
Frozen Orbs PERSON
DPS ORG
at least one CARDINAL
Spirit LOC
2 CARDINAL
2 CARDINAL
first ORDINAL
5-10% PERCENT
first ORDIN